# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook guides the exploration and processing of the FAIR² dataset using the `mlcroissant` library, referencing all fields and entities by their Croissant `@id` for clarity and reproducibility.

### Dataset Source
The FAIR² dataset is described using a Croissant schema and is accessible at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install -q mlcroissant pandas matplotlib

## 1. Data Loading

Load metadata and dataset records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Dataset object; access fields as attributes

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset identifier: {metadata.identifier}")
print(f"Version: {metadata.version}\n")

# Show selected high-level metadata fields
print("Keywords:", metadata.keywords)
print("Published:", metadata.datePublished)
print("Data license:", metadata.license)
print("Personal sensitive information:", getattr(metadata, 'personalSensitiveInformation', None))

## 2. Data Overview
Explore the available record sets and their Croissant `@id`, as well as available fields and columns within each record set. All references to record sets and fields use their unique `@id`.

In [ ]:
# List all RecordSets by their @id
print("All available record sets (by @id):")
for rs in dataset.record_sets:
    print(f"@id: {rs.id} | name: {rs.name}")

# For each RecordSet, list fields and columns by their @id
for rs in dataset.record_sets:
    print(f"\nRecordSet: {rs.name}")
    print(f"  @id: {rs.id}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    - name: {getattr(field, 'name', None)} | @id: {field.id}")
            if hasattr(field, 'columns'):
                for col in field.columns:
                    print(f"        Column: {getattr(col, 'name', None)} | @id: {col.id}")

## 3. Data Extraction

Extract data from a chosen record set, referencing entities by their `@id` only.

For this analysis, we select the main patient/provider record set, whose `@id` should correspond to the core clinical data (`@id`). (Below, you may want to inspect what comes out in section 2, and replace accordingly; for now, this will attempt to extract from the first record set.)

In [ ]:
# Prepare to load each record set into a pandas DataFrame
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set {record_set_id}, shape: {dataframes[record_set_id].shape}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Print the columns for the main record set (first available)
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for main record set (@id={main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No dataframes loaded.')

## 4. Exploratory Data Analysis (EDA)

Common EDA tasks include filtering, normalizing, and grouping data. All column/field accesses below reference the full Croissant `@id`.

Below, we:
- Select a numeric field (for example, patient age, if available) by its `@id`
- Filter out potentially implausible records
- Normalize the numeric column
- Group by an attribute (e.g., anatomical location)

Make sure to substitute the correct `@id` fields as found in your DataFrame from above.

In [ ]:
# Pick field @ids by inspecting DataFrame in the previous section
# For demonstration, we use the first numeric-looking field and a categorical field for grouping (replace as discovered)
main_df = dataframes[main_record_set_id]

# Let's try to auto-detect a numeric field and a group field by inspecting dtypes
numeric_field_id = None
group_field_id = None

for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col
        break

# Look for a non-numeric field
for col in main_df.columns:
    if not pd.api.types.is_numeric_dtype(main_df[col]):
        group_field_id = col
        break

print(f"Numeric field selected (@id): {numeric_field_id}")
print(f"Group field selected (@id): {group_field_id}\n")

# Set a threshold for filtering (e.g., > 10)
if numeric_field_id:
    threshold = 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and compare groups.

If the selected fields above are not ideal, modify to use a more relevant field based on your analysis in section 3.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id:
    plt.figure(figsize=(7,4))
    main_df[numeric_field_id].hist(bins=20, color="skyblue", edgecolor="black")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    if group_field_id in main_df.columns:
        plt.figure(figsize=(8,5))
        # Take top 10 group values by size (to avoid overcrowding)
        value_counts = main_df[group_field_id].value_counts().nlargest(10).index.tolist()
        sub_df = main_df[main_df[group_field_id].isin(value_counts)]
        sub_df.boxplot(column=numeric_field_id, by=group_field_id, grid=False, rot=60)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load, overview, process, and visualize the FAIR² dataset using the `mlcroissant` library, always referencing dataset entities by their Croissant `@id`.

- **Dataset explored:** Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors
- **Approach:**
    - Loaded the dataset metadata and identified record sets by their `@id`
    - Extracted tabular data for analysis
    - Performed basic EDA and visualization, focused on Croissant `@id` referencing

You can now use the extracted dataframes and Croissant `@id`s for further domain-specific analysis and for reproducibly referencing fields in downstream workflows.